##### Task 1: Preprocessing

In [5]:
import pandas as pd
import os

In [6]:
# CSV files 

file_path = "amazon_reviews.csv"

if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
    df = pd.read_csv(file_path,encoding="utf-8", on_bad_lines="skip")
    print("✅ CSV Loaded Successfully")
    print(df.head())
else:
    print("⚠️ File not found or empty")
    df = pd.DataFrame()

✅ CSV Loaded Successfully
         reviewer_name                                        review_text
0                   KT  This book breaks down the often-intimidating w...
1                laura            Easy to read and extremely informative!
2      Courtney Miller  My team references this book often when explai...
3  Chaminda Ranasinghe  This book gives a simple and methodical backgr...
4                mnowa  Presents mathematical concepts in an approacha...


In [7]:
# Check Data Info
print(df.info())      # Column info
print(df.shape)       # Rows, Columns
print(df.columns)     # Column names

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   reviewer_name  120 non-null    str  
 1   review_text    119 non-null    str  
dtypes: str(2)
memory usage: 2.0 KB
None
(120, 2)
Index(['reviewer_name', 'review_text'], dtype='str')


In [8]:
# Access Specific Columns
print(df["review_text"].head())

0    This book breaks down the often-intimidating w...
1              Easy to read and extremely informative!
2    My team references this book often when explai...
3    This book gives a simple and methodical backgr...
4    Presents mathematical concepts in an approacha...
Name: review_text, dtype: str


In [9]:
# Number of reviews
print("Total reviews:", len(df))

# Sample review
print(df.iloc[0]["review_text"])

Total reviews: 120
This book breaks down the often-intimidating world of data science into something approachable. The explanations are clear, and the examples are practical, making it perfect for beginners or anyone brushing up on their math skills. It’s a great resource for tackling the math side of data science with confidence


### Task 1: Preprocessing

In [7]:
## convert text to lowercase 
df["review_text"] = df["review_text"].astype(str).str.lower()
print(df["review_text"].head())

0    this book breaks down the often-intimidating w...
1              easy to read and extremely informative!
2    my team references this book often when explai...
3    this book gives a simple and methodical backgr...
4    presents mathematical concepts in an approacha...
Name: review_text, dtype: object


In [ ]:
### 1. Convert text to lowercase 
### 2 Tokenization 
### 3. Remove punctuation and non-alphabetic characters
### 4. Remove stop words (optional)
### 5. Lemmatization (optional)
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_tokenizer(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', str(text)) # remove non-alphabetic characters
    text = re.sub(r'[^\w\s]', '', text) # remove punctuation
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words]
    words = [w for w in words if w not in stop_words]
    tokens = re.findall(r'[a-z]+', ' '.join(words))   # only alphabets

    return tokens

df["tokens"] = df["review_text"].apply(clean_tokenizer)
print(df[["review_text", "tokens"]].head())

                                         review_text  \
0  this book breaks down the often-intimidating w...   
1            easy to read and extremely informative!   
2  my team references this book often when explai...   
3  this book gives a simple and methodical backgr...   
4  presents mathematical concepts in an approacha...   

                                              tokens  
0  [book, break, oftenintimidating, world, data, ...  
1               [easy, read, extremely, informative]  
2  [team, reference, book, often, explaining, for...  
3  [book, give, simple, methodical, background, u...  
4  [present, mathematical, concept, approachable,...  


#### Task 2: Vocabulary Creation

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

#vectorizer = CountVectorizer()

#X = vectorizer.fit_transform(df["tokens"])

# Vocabulary
#vocab = vectorizer.vocabulary_

#print("📊 Vocabulary Size:", len(vocab))

AttributeError: Module 'scipy' has no attribute '_lib'

: 

In [33]:
print(df.columns)

Index(['reviewer_name', 'review_text', 'tokens'], dtype='str')


In [ ]:
### Task 2: Vocabulary Creation

from collections import Counter
import numpy as np
# Flatten list of lists → single list
all_words = [word for tokens in df["tokens"] for word in tokens]

# Count frequency
word_freq = Counter(all_words)

vocab_2d = np.array(word_freq.most_common(20))

# Top 20 words
print(word_freq.most_common(20))

[('book', 264), ('data', 219), ('science', 112), ('interview', 88), ('question', 58), ('business', 55), ('concept', 45), ('chapter', 44), ('wa', 43), ('great', 41), ('math', 40), ('learning', 39), ('scientist', 34), ('topic', 34), ('like', 32), ('technical', 32), ('read', 31), ('author', 31), ('really', 30), ('one', 30)]


### Task 3: Feature Engineering


In [23]:
##### One Hot Encoding (document-level vector)
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False)
#encoder
encoder.fit(vocab_2d[:, 0].reshape(-1, 1))



OneHotEncoder(sparse_output=False)

In [21]:
encoder.categories_

[array(['author', 'book', 'business', 'chapter', 'concept', 'data',
        'great', 'interview', 'learning', 'like', 'math', 'one',
        'question', 'read', 'really', 'science', 'scientist', 'technical',
        'topic', 'wa'], dtype='<U11'),
 array(['112', '219', '264', '30', '31', '32', '34', '39', '40', '41',
        '43', '44', '45', '55', '58', '88'], dtype='<U11')]

In [17]:
print("Vocabulary:", encoder.categories_[0])

Vocabulary: ['author' 'book' 'business' 'chapter' 'concept' 'data' 'great' 'interview'
 'learning' 'like' 'math' 'one' 'question' 'read' 'really' 'science'
 'scientist' 'technical' 'topic' 'wa']


In [24]:
for word in vocab_2d[:, 0]:  # Only the words, not frequencies
    encoded_word = encoder.transform([[word]])
    print("\nWord:", word)
    print(encoded_word)


Word: book
[[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]

Word: data
[[0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]

Word: science
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]]

Word: interview
[[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]

Word: question
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]]

Word: business
[[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]

Word: concept
[[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]

Word: chapter
[[0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]

Word: wa
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]

Word: great
[[0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]

Word: math
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]

Word: learning
[[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]

Word: scientist
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.

In [29]:
##### Bag of Words using CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer
bow = CountVectorizer()

# Convert token lists to strings for CountVectorizer
df["processed_text"] = df["tokens"].apply(lambda x: ' '.join(x))
print(df["processed_text"].head())

0    book break oftenintimidating world data scienc...
1                      easy read extremely informative
2    team reference book often explaining formed co...
3    book give simple methodical background use mat...
4    present mathematical concept approachable way ...
Name: processed_text, dtype: object


In [37]:
bow_matrix = bow.fit_transform(df["processed_text"])
print("BOW Matrix shape:", bow_matrix.shape)
print("Vocabulary size:", len(bow.vocabulary_))
print("Sample BOW Matrix (first 5 documents):\n", bow_matrix.toarray()[:5])


BOW Matrix shape: (120, 2113)
Vocabulary size: 2113
Sample BOW Matrix (first 5 documents):
 [[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [32]:
bow.get_feature_names_out()

array(['abbie', 'ability', 'able', ..., 'youre', 'youve', 'zain'],
      dtype=object)

In [31]:
print("Vocabulary:", bow.get_feature_names_out())

Vocabulary: ['abbie' 'ability' 'able' ... 'youre' 'youve' 'zain']


In [38]:
##### TF-IDF using TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [39]:
tf_idf = TfidfVectorizer()

In [40]:
tf_idf_vector = tf_idf.fit_transform(df["processed_text"])

In [41]:
tf_idf_vector

<120x2113 sparse matrix of type '<class 'numpy.float64'>'
	with 5773 stored elements in Compressed Sparse Row format>

In [42]:
tf_idf_vector.toarray()

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.08114829, 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]])

In [43]:
for vector in tf_idf_vector.toarray():
    print(vector)

[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0.         0.         0.13110659 ... 0.         0.         0.        ]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0.         0.         0.         ... 0.09440784 0.14549877 0.        ]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0.         0.         0.         ... 0.34757017 0.         0.        ]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0.

In [44]:

tf_idf.get_feature_names_out()

array(['abbie', 'ability', 'able', ..., 'youre', 'youve', 'zain'],
      dtype=object)

In [45]:
print("Vocabulary:", tf_idf.get_feature_names_out())

Vocabulary: ['abbie' 'ability' 'able' ... 'youre' 'youve' 'zain']


In [70]:
vectorizer_tfidf = TfidfVectorizer()
X_tfidf = vectorizer_tfidf.fit_transform(df["processed_text"])

tfidf_df = pd.DataFrame(X_tfidf.toarray(),
                        columns=vectorizer_tfidf.get_feature_names_out())

print("\n🔹 TF-IDF")
print(tfidf_df)


🔹 TF-IDF
     abbie  ability  able  absolute  absolutely  abundant  academia  academic  \
0      0.0      0.0   0.0       0.0         0.0       0.0       0.0       0.0   
1      0.0      0.0   0.0       0.0         0.0       0.0       0.0       0.0   
2      0.0      0.0   0.0       0.0         0.0       0.0       0.0       0.0   
3      0.0      0.0   0.0       0.0         0.0       0.0       0.0       0.0   
4      0.0      0.0   0.0       0.0         0.0       0.0       0.0       0.0   
..     ...      ...   ...       ...         ...       ...       ...       ...   
115    0.0      0.0   0.0       0.0         0.0       0.0       0.0       0.0   
116    0.0      0.0   0.0       0.0         0.0       0.0       0.0       0.0   
117    0.0      0.0   0.0       0.0         0.0       0.0       0.0       0.0   
118    0.0      0.0   0.0       0.0         0.0       0.0       0.0       0.0   
119    0.0      0.0   0.0       0.0         0.0       0.0       0.0       0.0   

     academically

In [47]:
tfidf_df = pd.DataFrame(tf_idf_vector.toarray(),
                        columns=tf_idf.get_feature_names_out())

print("\n🔹 TF-IDF")
print(tfidf_df)


🔹 TF-IDF
     abbie  ability  able  absolute  absolutely  abundant  academia  academic  \
0      0.0      0.0   0.0       0.0         0.0       0.0       0.0       0.0   
1      0.0      0.0   0.0       0.0         0.0       0.0       0.0       0.0   
2      0.0      0.0   0.0       0.0         0.0       0.0       0.0       0.0   
3      0.0      0.0   0.0       0.0         0.0       0.0       0.0       0.0   
4      0.0      0.0   0.0       0.0         0.0       0.0       0.0       0.0   
..     ...      ...   ...       ...         ...       ...       ...       ...   
115    0.0      0.0   0.0       0.0         0.0       0.0       0.0       0.0   
116    0.0      0.0   0.0       0.0         0.0       0.0       0.0       0.0   
117    0.0      0.0   0.0       0.0         0.0       0.0       0.0       0.0   
118    0.0      0.0   0.0       0.0         0.0       0.0       0.0       0.0   
119    0.0      0.0   0.0       0.0         0.0       0.0       0.0       0.0   

     academically

In [57]:
vectorizer_bow = CountVectorizer()
X_bow = vectorizer_bow.fit_transform(df["processed_text"])

bow_df = pd.DataFrame(X_bow.toarray(),
                      columns=vectorizer_bow.get_feature_names_out())

print("\n🔹 Bag of Words (BoW)")
print(bow_df)


🔹 Bag of Words (BoW)
     abbie  ability  able  absolute  absolutely  abundant  academia  academic  \
0        0        0     0         0           0         0         0         0   
1        0        0     0         0           0         0         0         0   
2        0        0     0         0           0         0         0         0   
3        0        0     0         0           0         0         0         0   
4        0        0     0         0           0         0         0         0   
..     ...      ...   ...       ...         ...       ...       ...       ...   
115      0        0     0         0           0         0         0         0   
116      0        0     0         0           0         0         0         0   
117      0        0     0         0           0         0         0         0   
118      0        0     0         0           0         0         0         0   
119      0        0     0         0           0         0         0         0   

     

In [58]:
# Convert each word into binary presence/absence
vectorizer_ohe = CountVectorizer(binary=True)
X_ohe = vectorizer_ohe.fit_transform(df["processed_text"])

ohe_df = pd.DataFrame(X_ohe.toarray(),
                       columns=vectorizer_ohe.get_feature_names_out())

print("🔹 One Hot Encoding (OHE)")
print(ohe_df)

🔹 One Hot Encoding (OHE)
     abbie  ability  able  absolute  absolutely  abundant  academia  academic  \
0        0        0     0         0           0         0         0         0   
1        0        0     0         0           0         0         0         0   
2        0        0     0         0           0         0         0         0   
3        0        0     0         0           0         0         0         0   
4        0        0     0         0           0         0         0         0   
..     ...      ...   ...       ...         ...       ...       ...       ...   
115      0        0     0         0           0         0         0         0   
116      0        0     0         0           0         0         0         0   
117      0        0     0         0           0         0         0         0   
118      0        0     0         0           0         0         0         0   
119      0        0     0         0           0         0         0         0   

  

#### Task 4: Comparison Analysis


In [71]:
comparison = pd.concat(
    {
        "OHE": ohe_df,
        "BoW": bow_df,
        "TF-IDF": tfidf_df
    },
    axis=1
)

print("\n🔹 Comparison Table")
print(comparison)


🔹 Comparison Table
      OHE                                                              \
    abbie ability able absolute absolutely abundant academia academic   
0       0       0    0        0          0        0        0        0   
1       0       0    0        0          0        0        0        0   
2       0       0    0        0          0        0        0        0   
3       0       0    0        0          0        0        0        0   
4       0       0    0        0          0        0        0        0   
..    ...     ...  ...      ...        ...      ...      ...      ...   
115     0       0    0        0          0        0        0        0   
116     0       0    0        0          0        0        0        0   
117     0       0    0        0          0        0        0        0   
118     0       0    0        0          0        0        0        0   
119     0       0    0        0          0        0        0        0   

                              

In [72]:
import numpy as np

# Average TF-IDF score per word
avg_tfidf = tfidf_df.mean(axis=0).sort_values(ascending=False)

print("\n🔹 Most Important Words (TF-IDF Ranking)")
print(avg_tfidf)


🔹 Most Important Words (TF-IDF Ranking)
book           0.087803
data           0.076446
science        0.051599
interview      0.048533
math           0.034820
                 ...   
nosql          0.000329
sa             0.000329
computation    0.000329
john           0.000329
optimistic     0.000329
Length: 2113, dtype: float64


#### Task 5: Sparse Matrix Analysis

In [73]:
print("OHE Shape:", X_ohe.shape)
print("BoW Shape:", X_bow.shape)
print("TF-IDF Shape:", X_tfidf.shape)

OHE Shape: (120, 2113)
BoW Shape: (120, 2113)
TF-IDF Shape: (120, 2113)


In [74]:
import numpy as np

def calculate_sparsity(matrix):
    total_elements = matrix.shape[0] * matrix.shape[1]
    non_zero = matrix.count_nonzero()
    zero_elements = total_elements - non_zero
    sparsity = (zero_elements / total_elements) * 100
    return sparsity

print("OHE Sparsity: {:.2f}%".format(calculate_sparsity(X_ohe)))
print("BoW Sparsity: {:.2f}%".format(calculate_sparsity(X_bow)))
print("TF-IDF Sparsity: {:.2f}%".format(calculate_sparsity(X_tfidf)))

OHE Sparsity: 97.72%
BoW Sparsity: 97.72%
TF-IDF Sparsity: 97.72%
